In [1]:
import random
from datetime import timedelta, date
from tqdm import tqdm

from src.minio_utils import upload_df_to_s3
from src.generate_data import (
    get_master_data, save_master_data,
    generate_new_customers, generate_new_products,
    simulate_scd_updates, generate_daily_orders
)

def main():
    print("--- Starting Stateful Historical Generation ---")
    
    # 1. Initialize empty or load existing master DFs
    # We load these from local CSVs so we "remember" customers from day to day
    df_customers = get_master_data('customers.csv', ['customer_id', 'signup_date', 'city'])
    df_products = get_master_data('products.csv', ['product_id', 'unit_price'])

    # 2. Define the Time Range (2 Years ago -> Yesterday)
    start_date = date.today() - timedelta(days=730)
    end_date = date.today() - timedelta(days=30)
    
    total_days = (end_date - start_date).days + 1
    
    # 3. The Daily Loop
    with tqdm(total=total_days, desc="Simulating History") as pbar:
        current_date = start_date
        while current_date <= end_date:
            date_str = current_date.strftime("%Y-%m-%d")
            pbar.set_postfix(date=date_str)
            
            # --- A. EVOLVE THE DATA (SCD & New Signups) ---
            
            # 1. New Signups: Add 2-5 new customers every single day
            df_customers = generate_new_customers(df_customers, date_str, num_new=random.randint(2, 5))
            
            # 2. New Products: Add 1 new product occasionally (10% chance)
            if random.random() < 0.1:
                df_products = generate_new_products(df_products, num_new=1)
                
            # 3. SCD Type 2: 2 existing customers move to a new city today
            # The function updates the 'city' column in df_customers in-place
            df_customers = simulate_scd_updates(df_customers, date_str, num_updates=2)
            
            # --- B. GENERATE TRANSACTIONS ---
            
            # Random chaos on the 15th of the month
            chaos = random.randint(5, 25) / 100 if current_date.day % 4 == 0 else 0.0
            
            # Generate orders using the CURRENT state of customers (including today's new ones)
            df_orders, df_items = generate_daily_orders(
                date_str, df_products, df_customers, 
                num_orders=random.randint(20, 50), chaos_rate=chaos
            )
            
            # --- C. SAVE SNAPSHOTS TO S3 (Bronze Layer) ---
            
            # We upload the FULL customer list every day. 
            # This allows the Silver layer (DBT) to compare "Yesterday vs Today" 
            # to detect the SCD changes we just made.
            upload_df_to_s3(df_customers, 'customers', date_str, is_local=True)
            upload_df_to_s3(df_products, 'products', date_str, is_local=True)
            upload_df_to_s3(df_orders, 'orders', date_str, is_local=True)
            upload_df_to_s3(df_items, 'order_items', date_str, is_local=True)
            
            current_date += timedelta(days=1)
            pbar.update(1)
    
    # 4. Save Final State to Local Disk
    # This ensures your Airflow DAG can pick up exactly where this script left off
    print("Saving final master state to src/data/...")
    save_master_data(df_customers, 'customers.csv')
    save_master_data(df_products, 'products.csv')
    print("Done! You can now turn on your Airflow DAG.")

if __name__ == "__main__":
    main()

--- Starting Stateful Historical Generation ---


Simulating History: 100%|██████████| 701/701 [04:00<00:00,  2.92it/s, date=2026-01-15]

Saving final master state to src/data/...
Done! You can now turn on your Airflow DAG.
